# 09 - Full BGL dataset

Parse the full BGL log with Drain3 and cut it into windows of 20 lines, the same as the week 1 test on the 2k sample.

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.features.bgl_features import EMPTY_MESSAGE, build_bgl

pd.set_option("display.max_colwidth", 120)

## Parse and build windows

This reads `BGL.log` line by line. A window is an anomaly if any of its 20 lines is an alert (label other than `-`).

In [2]:
sequences, counts, stats = build_bgl(window_size=20)

print(f"Total lines:       {stats['lines']:,}")
print(f"Unparsed lines:    {stats['unparsed_lines']:,}")
print(f"Alert lines:       {stats['alert_lines']:,} ({stats['alert_lines'] / stats['lines']:.2%})")
print(f"Templates:         {stats['templates']:,}")
print(f"Windows:           {stats['windows']:,}")
print(f"Anomalous windows: {stats['anomalous_windows']:,} ({stats['anomalous_windows'] / stats['windows']:.2%})")
print(f"Total time:        {stats['total_seconds'] / 60:.1f} min")

Parsing BGL.log: 0 lines [00:00, ? lines/s]

Parsing BGL.log: 1748540 lines [00:30, 58284.65 lines/s]

Parsing BGL.log: 1748540 lines [00:40, 58284.65 lines/s]

Parsing BGL.log: 3396851 lines [01:00, 56319.37 lines/s]

Parsing BGL.log: 3396851 lines [01:11, 56319.37 lines/s]

Parsing BGL.log: 4404509 lines [01:30, 45939.97 lines/s]

Parsing BGL.log: 4404509 lines [01:42, 45939.97 lines/s]

Parsing BGL.log: 4747963 lines [01:46, 44707.07 lines/s]

Total lines:       4,747,963
Unparsed lines:    0
Alert lines:       348,460 (7.34%)
Templates:         391
Windows:           237,398
Anomalous windows: 20,261 (8.53%)
Total time:        1.8 min


## Example sequences

In [3]:
# two normal windows and one anomaly
examples = pd.concat([sequences[sequences["Label"] == 0].head(2), sequences[sequences["Label"] == 1].head(1)])
for row in examples.itertuples():
    label = "anomaly" if row.Label else "normal"
    print(f"window {row.window} ({label}): {' '.join(row.EventSequence)}")

window 0 (normal): E1 E1 E1 E1 E1 E1 E1 E1 E1 E1 E1 E1 E1 E1 E1 E1 E1 E1 E1 E1
window 1 (normal): E1 E1 E1 E1 E1 E1 E1 E1 E1 E1 E1 E1 E1 E1 E1 E1 E1 E1 E1 E1
window 736 (anomaly): E6 E7 E7 E7 E2 E2 E2 E2 E13 E14 E15 E2 E2 E2 E2 E16 E16 E17 E17 E17


## Check: empty messages

Some BGL lines have only the 9 header fields and no message. They are parsed as their own event, `EMPTY_MESSAGE`, so they stay in their windows and nothing is counted as unparsed.

In [4]:
templates = pd.read_csv(ROOT / "data" / "processed" / "bgl_templates.csv").set_index("EventId")["template"]
empty_id = templates.index[templates == EMPTY_MESSAGE][0]
n_empty, n_empty_alerts = stats["empty_message_lines"], stats["empty_message_alerts"]

print(f"Unparsed lines: {stats['unparsed_lines']:,}")
print(f"EMPTY_MESSAGE lines ({empty_id}): {n_empty:,}, alerts: {n_empty_alerts:,}, normal: {n_empty - n_empty_alerts:,}")
print(f"{empty_id} count over all windows: {counts[empty_id].sum():,}")

Unparsed lines: 0
EMPTY_MESSAGE lines (E198): 34,470, alerts: 0, normal: 34,470
E198 count over all windows: 34,470


## Check: masking before and after

The BGL masks turn hex words without `0x` into `<HEX>` and register numbers (`fpr12=`, `r07=`) into `<NUM>`. "Before" re-parses the log with the shared masks only, without saving.

In [5]:
def template_summary(counts, n_templates):
    freq = counts.filter(regex=r"^E\d+$").sum()
    return {"templates": n_templates,
            "templates seen < 10 times": int((freq < 10).sum()),
            "lines covered by top 50": f"{freq.nlargest(50).sum() / freq.sum():.2%}"}


_, counts_before, stats_before = build_bgl(window_size=20, use_dataset_masks=False, save=False)
summary = pd.DataFrame({"before (shared masks)": template_summary(counts_before, stats_before["templates"]),
                        "after (+ BGL masks)": template_summary(counts, stats["templates"])})
del counts_before
summary

Parsing BGL.log: 0 lines [00:00, ? lines/s]

Parsing BGL.log: 2493169 lines [00:30, 83105.61 lines/s]

Parsing BGL.log: 2493169 lines [00:42, 83105.61 lines/s]

Parsing BGL.log: 4459121 lines [01:00, 72768.02 lines/s]

Parsing BGL.log: 4747963 lines [01:07, 70284.96 lines/s]

,before (shared masks),after (+ BGL masks)
templates,1124,391
templates seen < 10 times,762,79
lines covered by top 50,94.14%,94.14%


10 of the rarest templates after the new masks. Values that still change from line to line show which fields are not masked yet.

In [6]:
freq = counts.filter(regex=r"^E\d+$").sum()
rare = freq[freq > 0].sort_values(kind="stable").head(10)
pd.DataFrame({"count": rare, "template": rare.index.map(templates)})

,count,template
E165,1,NodeCard VPD chip is not accessible
E166,1,"While initializing node card, ido with LP of FF:F2:9F:<NUM>:E6:<NUM>:<NUM>:0D:<NUM>:E9:<NUM>:<NUM> caught java.lang...."
E167,1,"Node card status: ALERT <NUM>, ALERT <NUM>, ALERT <NUM>, ALERT <NUM> is (are) active. Clock Mode is Low. Clock Selec..."
E175,1,"e error threshold, consider replacing the card"
E190,1,parity error.......<NUM>
E191,1,machine state register:
E209,1,MailboxMonitor::serviceMailboxes() lib_ido_error: <NUM> unexpected socket error: Broken pipe
E219,1,<NUM>:fffffefa <NUM>:<HEX> <NUM>:<HEX> <NUM>:<HEX>
E224,1,DeclareServiceNetworkCharacteristics has been run but the DB is not empty
E225,1,DeclareServiceNetworkCharacteristics has been run with the force option but the DB is not empty
